# Лабораторная работа 3. Вариант А.

In [121]:
import math

import numpy as np

rng = np.random.default_rng(580497)

def chi2_cdf(x, df):
    if x <= 0:
        return 0.0
    a, z = df / 2, x / 2
    P, a_cur = math.erf(math.sqrt(z)), 0.5
    while a_cur < a - 1e-12:
        P -= z**a_cur * math.exp(-z) / math.gamma(a_cur + 1)
        a_cur += 1.0
    return P


def normal_cdf(x, mu, sigma):
    return 0.5 * (1 + math.erf((x - mu) / (sigma * math.sqrt(2))))


def laplace_cdf(x, mu, b):
    return 0.5 * math.exp((x - mu) / b) if x < mu else 1 - 0.5 * math.exp(-(x - mu) / b)


## Задача 1

Для реализаций совокупности наблюдений $(\nu_1, \dots, \nu_4)$, имеющих полиномиальное распределение $\Pi(n, p_*)$, проверьте простую гипотезу о вероятностях
$$
    H_0: p_* = p_0
$$
по критерию хи-квадрат со статистикой критерия:
$$
    X^2(\nu;p_0)
    = \sum \limits_{i=1}^k \frac{(\nu_i - n p_{0,i})^2}{n p_{0,i}} .
$$
В качестве критической области $\Gamma_\alpha$ возьмите правостороннюю область:
$$
    \Gamma_\alpha = (h_\alpha; +\infty), \\
    h_\alpha = F_{\chi^2}^{-1}(1 - \alpha; 3) ,
$$
где $\alpha$ - уровень значимости и $F_{\chi^2}^{-1}(p;3)$ - квантиль уровня $p$ распределения хи-квадрат с 3 степенями свободы.

In [122]:
def quantile(p):
    lo, hi = 0.0, 50.0
    for _ in range(100):
        mid = (lo + hi) / 2
        if chi2_cdf(mid, df=3) < p:
            lo = mid
        else:
            hi = mid
    return (lo + hi) / 2

def build_run(p0, n, l):
    def run(p_star, alpha, accept):
        h = quantile(1 - alpha)
        nu = rng.multinomial(n, p_star, size=l)  # (l, k)
        X2 = ((nu - n * p0) ** 2 / (n * p0)).sum(axis=1)  # (l,)
        hits = (X2 < h) if accept else (X2 >= h)
        return hits.sum(), X2, h

    return run

1. Сформируйте $l=10$ реализаций совокупности наблюдений $(\nu_1, \dots, \nu_4)$ для $n=100$ и вектора вероятностей $p_*=(0.2,0.45,0.15,0.2)$. Проверьте гипотезу
с вектором вероятностей $p_0 = (0.2,0.45,0.15,0.2)$ и уровнем значимости $\alpha = 0.05$. Подсчитайте, сколько раз из общего количества $l$ критерием была принята верная гипотеза $H_0$.

In [123]:
# 1) верная H0, α = 0.05
run = build_run(
    p0=np.array([0.2, 0.45, 0.15, 0.2]),  # .
    n=100,
    l=10,
)
acc, s1, h1 = run(
    p_star=np.array([0.2, 0.45, 0.15, 0.2]),  # .
    alpha=0.05,
    accept=True,
)
print(f"H0 верна:\n\tпринята {acc}/{10}, h={h1:.2f}, X²={np.round(s1, 2)}")

H0 верна:
	принята 9/10, h=7.81, X²=[7.92 3.57 2.61 0.32 1.66 4.46 1.74 2.85 7.39 2.32]


2. Сформируйте $l$ = 10 реализаций совокупности наблюдений $(\nu_1, \dots, \nu_4)$ для $n=100$ и вектора вероятностей $p_*=(0.4,0.25,0.15,0.2)$. Проверьте
гипотезу с вектором вероятностей $p_0 = (0.2,0.45,0.15,0.2)$ и уровнем значимости $\alpha = 0.1$. Подсчитайте, сколько раз из общего количества $l$ критерием была отклонена неверная гипотеза $H_0$.

In [124]:
# 2) неверная H0, α = 0.1
run = build_run(
    p0=np.array([0.2, 0.45, 0.15, 0.2]),  # .
    n=100,
    l=10,
)
rej, s1, h1 = run(
    p_star=np.array([0.4, 0.25, 0.15, 0.2]),  # .
    alpha=0.1,
    accept=False,
)
print(f"H0 неверна:\n\tНепринята {rej}/{10}, h={h1:.2f}, X²={np.round(s1, 2)}")

H0 неверна:
	Непринята 10/10, h=6.25, X²=[43.76 42.81 30.19 47.07 13.32 21.57 34.16 20.29 29.36 18.62]


## Задача 2

Сформируйте $l=10$ реализаций совокупности наблюдений $\nu = (\nu_1, \dots, \nu_4)$, имеющих полиномиальное распределение $\Pi(n, p_*)$, с $n=70$
и вектором вероятностей $p_*=(0.3, 0.2, 0.35, 0.15)$. Для каждой реализации найдите наименьший уровень значимости $\alpha$, при котором отклоняется основная гипотеза
$H_0$ с вектором вероятностей $p_0=(0.3,0.2,0.35,0.15)$:
$$
    \alpha = 1 - F_{\chi^2}(X^2(\nu;p_0); 3), \\
    X^2(\nu;p_0) = \sum \limits_{i=1}^k \frac{(\nu_i - n p_{0,i})^2}{n p_{0,i}} ,
$$
где $F_{\chi^2}(x;3)$ - значение функции распределения для распределения хи-квадрат с 3 степенями свободы.

In [125]:
n, l = 70, 10
p0 = np.array([0.3, 0.2, 0.35, 0.15])
p_star = np.array([0.3, 0.2, 0.35, 0.15])

for j in range(1, l + 1):
    nu = rng.multinomial(n, p_star) 
    X2 = np.sum((nu - n * p0) ** 2 / (n * p0)) 
    alpha = 1 - chi2_cdf(X2, df=3)
    print(f"#{j}: nu={nu.tolist()}, X²={X2:.3f}, α={alpha:.4f}")

#1: nu=[19, 16, 24, 11], X²=0.510, α=0.9166
#2: nu=[17, 16, 22, 15], X²=3.231, α=0.3573
#3: nu=[13, 15, 30, 12], X²=4.568, α=0.2063
#4: nu=[17, 15, 27, 11], X²=1.112, α=0.7741
#5: nu=[21, 16, 24, 9], X²=0.510, α=0.9166
#6: nu=[23, 18, 22, 7], X²=2.755, α=0.4309
#7: nu=[22, 14, 24, 10], X²=0.082, α=0.9939
#8: nu=[33, 8, 21, 8], X²=10.524, α=0.0146
#9: nu=[20, 15, 21, 14], X²=1.786, α=0.6180
#10: nu=[21, 11, 24, 14], X²=1.820, α=0.6107


## Задача 3

Для двух дискретных случайных величин $\xi$ и $\eta$, принимающих конечное количество значений $\{1, \dots, k\}$ и $\{1, \dots, m\}$ сооветветственно,
сформируйте реализацию выборки из пар $(\xi_i, \eta_i)$ объёма $n$ и вычислите реализацию совокупности наблюдений
$\nu = (\nu_{11}, \dots, \nu_{km})$, где $\nu_{ij}$ - количество пар, в которых $\xi = i$ и $\eta = j$.

Проверьте гипотезу о независимости случайных величин $\xi$ и $\eta$ по критерию хи-квадрат со статистикой:
$$
    X^2(\nu;\widehat{\theta})
    = \sum \limits_{i=1}^k \sum \limits_{j=1}^m
    \frac{(\nu_{ij} - n \widehat{\theta}_{\xi,i} \widehat{\theta}_{\eta,j})^2}{n \widehat{\theta}_{\xi,i} \widehat{\theta}_{\eta,j}} , \\
    %
    \widehat{\theta}_{\xi,i} = \frac{\sum \limits_{j=1}^m \nu_{ij}}{n} , \\
    \widehat{\theta}_{\eta,j} = \frac{\sum \limits_{i=1}^k \nu_{ij}}{n} ,
$$
определив наименьший уровень значимости отклонения гипотезы о независимости:
$$
    \alpha = 1 - F_{\chi^2}(X^2(\nu;\widehat{\theta}); (k-1)(m-1)) ,
$$
где $F_{\chi^2}(X^2(\nu;\widehat{\theta}); (k-1)(m-1))$ - значение функции распределения для распределения хи-квадрат с $(k-1)(m-1)$ степенями свободы.

In [126]:
def X2_indep(nu, n):
    k, m = nu.shape
    th_xi = nu.sum(axis=1) / n
    th_eta = nu.sum(axis=0) / n
    exp = n * np.outer(th_xi, th_eta)
    X2 = np.sum((nu - exp) ** 2 / exp)
    return X2, (k - 1) * (m - 1)


def check(name, pairs, n):
    nu = np.zeros((3, 2), dtype=int)
    for xi, eta in pairs:
        nu[xi - 1, eta - 1] += 1
    X2, df = X2_indep(nu, n)
    alpha = 1 - chi2_cdf(X2, df)
    print(f"{name}")
    print(f"  ν = {nu.tolist()}")
    print(f"  X² = {X2:.3f},  df = {df},  α = {alpha:.4f}")


1. Сформируйте $n=30$ реализаций пар случайных величин $(\xi, \eta)$, в которых величина $\xi$ принимает с равной вероятностью одно из значений $\{1, 2, 3\}$,
а случайная величина $\eta$ не зависимо от неё с равной вероятностью принимает одно из значений $\{1, 2\}$.

In [127]:
n = 30

# 1) независимость: xi - U{1,2,3}, eta - U{1,2}
pairs1 = [(rng.integers(1, 4), rng.integers(1, 3)) for _ in range(n)]
check("1) ξ не зависит от η", pairs1, n)


1) ξ не зависит от η
  ν = [[5, 3], [6, 9], [5, 2]]
  X² = 2.262,  df = 2,  α = 0.5198


2. Сформируйте $n=30$ реализаций пар случайных величин $(\xi, \eta)$, в которых величина $\xi$ принимает с равной вероятностью одно из значений ${1, 2, 3}$,
а случайная величина $\eta$:
- принимает значение 1 с вероятностью 0.8 и 2 с вероятностью 0.2, если $\xi$ приняло значение 1 или 2,
- принимает значение 1 с вероятностью 0.3 и 2 с вероятностью 0.7, если $\xi$ приняло значение 3.

In [ ]:
# 2) зависимость: η от ξ
pairs2 = []
for _ in range(n):
    xi = rng.integers(1, 4)
    p1 = 0.8 if xi in (1, 2) else 0.3
    eta = 1 if rng.random() < p1 else 2
    pairs2.append((xi, eta))
check("2) η зависит от ξ", pairs2, n)

2) η зависит от ξ
  ν = [[9, 1], [11, 0], [2, 7]]
  X² = 17.443,  df = 2,  α = 0.0006


## Задача 4

Для $k=3$ дискретных величин $\xi_1$, $\xi_2$ и $\xi_3$, принимающих конечное количество значений $\{1, \dots, m\}$, сформируйте выборки объёмами $n_1$, $n_2$ и $n_3$, посчитайте количества $\nu_{ij}$, в которых величина $\xi_i$ принимает значение $j$.

Проверьте гипотезу об однородности распределений случайных величин по критерию со статистикой:
$$
    X^2(\nu;\widehat{\theta})
    = \sum \limits_{i=1}^k \sum \limits_{j=1}^m
    \frac{(\nu_{ij} - n \widehat{\theta}_j)^2}{n \widehat{\theta}_j} , \\
    %
    \widehat{\theta}_j = \frac{\sum \limits_{i=1}^k \nu_{ij}}{n} ,
$$
вычислив наименьший уровень значимости $\alpha$, при котором отклоняется гипотеза об однородности:
$$
    \alpha = 1 - F_{\chi^2}(X^2(\nu;\widehat{\theta}); (k-1)(m-1)) ,
$$
где $F_{\chi^2}(X^2(\nu;\widehat{\theta}); (k-1)(m-1))$ - значение функции распределения для распределения хи-квадрат с $(k-1)(m-1)$ степенями свободы.


In [129]:
def X2_homog(nu, n_i, n):
    # nu: k * m счётчиков, n_i: объёмы выборок, n = sum n_i
    m = nu.shape[1]
    th = nu.sum(axis=0) / n
    exp = np.outer(n_i, th)  # ожидания n_i * theta'_j
    X2 = np.sum((nu - exp) ** 2 / exp)
    return X2, (nu.shape[0] - 1) * (m - 1)


def check(name, probs_list, n_list):
    k, m = len(probs_list), len(probs_list[0])
    nu = np.array([rng.multinomial(n, p) for n, p in zip(n_list, probs_list)])
    n = sum(n_list)
    X2, df = X2_homog(nu, np.array(n_list), n)
    alpha = 1 - chi2_cdf(X2, df)
    print(f"{name}")
    print(f"  ν = {nu.tolist()}")
    print(f"  X² = {X2:.3f},  df = {df},  α = {alpha:.4f}")


1. $n_1 = 20$, $n_2 = 30$, $n_3 = 25$, распределения величин
-
| $\xi_1$ | 1    | 2    | 3    | 4    |
|---------|------|------|------|------|
| $p$     | 0.2  | 0.4  | 0.2  | 0.2  |

-
| $\xi_1$ | 1    | 2    | 3    | 4    |
|---------|------|------|------|------|
| $p$     | 0.23 | 0.38 | 0.17 | 0.22 |

-
| $\xi_1$ | 1    | 2    | 3    | 4    |
|---------|------|------|------|------|
| $p$     | 0.19 | 0.38 | 0.22 | 0.21 |


In [130]:
# 1) почти однородные распределения
n1, n2, n3 = 20, 30, 25
ns = [n1, n2, n3]
check("1) почти однородные",
      [[0.2, 0.4, 0.2, 0.2],
       [0.23, 0.38, 0.17, 0.22],
       [0.19, 0.38, 0.22, 0.21]], ns)

1) почти однородные
  ν = [[4, 8, 3, 5], [4, 17, 5, 4], [4, 10, 7, 4]]
  X² = 3.602,  df = 6,  α = 0.8243


2. $n_1 = 20$, $n_2 = 30$, $n_3 = 25$, распределения величин
-
| $\xi_1$ | 1    | 2    | 3    | 4    |
|---------|------|------|------|------|
| $p$     | 0.2  | 0.4  | 0.2  | 0.2  |

-
| $\xi_1$ | 1    | 2    | 3    | 4    |
|---------|------|------|------|------|
| $p$     | 0.2  | 0.2  | 0.4  | 0.2  |

-
| $\xi_1$ | 1    | 2    | 3    | 4    |
|---------|------|------|------|------|
| $p$     | 0.2  | 0.4  | 0.2  | 0.2  |


In [131]:
# 2) вторая выборка резко смещена
n1, n2, n3 = 20, 30, 25
ns = [n1, n2, n3]
check("2) есть смещённая выборка",
      [[0.2, 0.4, 0.2, 0.2],
       [0.2, 0.2, 0.4, 0.2],
       [0.2, 0.4, 0.2, 0.2]], ns)

2) есть смещённая выборка
  ν = [[4, 7, 3, 6], [5, 4, 15, 6], [4, 13, 2, 6]]
  X² = 17.476,  df = 6,  α = 0.0146


## Задача 5

Сформируйте реализацию выборки $\zeta = (\xi_1, \dots, \xi_n)$ с функцией распределения $F_*(x)$ и проверьте основную гипотезу о распределении
$$
    H_0: F_*(x) = F_0(x)
$$
по критерию согласия Колмогорова со статистикой:
$$
    D_n = \sqrt{n} \sup \limits_{x} |F_n^*(x; \zeta) - F_0(x)| .
$$
Найдите наименьший уровень значимости отклонения гипотезы:
$$
    \alpha = 1 - F(D_n(\zeta)),
$$
используя точную функцию распределения $F(x)$ статистики $D_n(\zeta)$ или асимптотическое распределение Колмогорова $K(t)$.

In [132]:


def Dn_stat(sample, F0, n):
    x = np.sort(sample)
    F0v = np.array([F0(v) for v in x])
    D = 0.0
    for i in range(n):
        D = max(D, abs((i + 1) / n - F0v[i]), abs(i / n - F0v[i]))
    return math.sqrt(n) * D


def kolmogorov_cdf_exact(t, n):
    # P(D_n < t) = n!/n^n * det(H), H — n×n, индексы 1..n:
    #   H[i][j] = 1,                              если i = j+1
    #   H[i][j] = ((j-i+1)t)^{j-i+1}/(j-i+1)!,    если i <= j
    #   H[i][j] = 0,                              иначе
    if t <= 0: return 0.0
    if t >= 1: return 1.0
    H = [[0.0]*n for _ in range(n)]
    for i in range(n):            # 0-based i ↔ 1-based i+1
        for j in range(n):        # 0-based j ↔ 1-based j+1
            ii, jj = i + 1, j + 1
            if ii == jj + 1:                  # i = j+1 в 1-based
                H[i][j] = 1.0
            elif ii <= jj:                    # i <= j в 1-based
                r = jj - ii + 1
                H[i][j] = (r * t) ** r / math.factorial(r)
    # det(H) методом Гаусса
    A = [row[:] for row in H]
    det = 1.0
    for i in range(n):
        p = max(range(i, n), key=lambda r: abs(A[r][i]))
        if abs(A[p][i]) < 1e-300:
            return 0.0
        if p != i:
            A[i], A[p] = A[p], A[i]
            det = -det
        det *= A[i][i]
        for r in range(i+1, n):
            f = A[r][i] / A[i][i]
            for c in range(i, n):
                A[r][c] -= f * A[i][c]
    return math.factorial(n) / (n**n) * abs(det)


def kolmogorov_cdf_asymp(t):
    if t <= 0:
        return 0.0
    s = 0.0
    for k in range(-50, 51):
        s += (-1) ** k * math.exp(-2 * k * k * t * t)
    return s


def run_case(name, n, true_dist, F0, cdf_D):
    sample = []
    for _ in range(n):
        if true_dist == "normal":
            # Box-Muller, N(-1, 2)
            u1 = rng.random() + 1e-300
            u2 = rng.random()
            z = math.sqrt(-2 * math.log(u1)) * math.cos(2 * math.pi * u2)
            sample.append(-1 + 2 * z)
        elif true_dist == "laplace":
            u = rng.random()
            sample.append(
                -1 + math.log(2 * u) if u < 0.5 else -1 - math.log(2 * (1 - u))
            )
    D = Dn_stat(sample, F0, n)
    alpha = 1 - cdf_D(D)
    print(
        f"{name}: n={n}, Dₙ={D:.4f}, α={alpha:.4f}"
    )


1. Объём $n = 10$, выборка из нормального распределения $\mathcal{N}(-1,2)$,
- распределение основной гипотезы $H_0$ - нормальное распределение $\mathcal{N}(-1,2)$,
- распределение основной гипотезы $H_0$ - распределение Лапласа $L(-1,1)$ с плотностью:
$$
    p(x)
    = \frac{1}{2} e^{-|x+1|}.
$$
Использовать точное распределение статистики.

In [133]:
F0_normal  = lambda x: normal_cdf(x, -1, 2)
F0_laplace = lambda x: laplace_cdf(x, -1, 1)

print("n=10, точное F(D_n)")
run_case("1a) H0 верна: выборка N(-1,2), F0=N(-1,2)", 10, 'normal', F0_normal,
         lambda t: kolmogorov_cdf_exact(t, 10))
run_case("1b) H0 неверна: выборка N(-1,2), F0=L(-1,1)", 10, 'normal', F0_laplace,
         lambda t: kolmogorov_cdf_exact(t, 10))

n=10, точное F(D_n)
1a) H0 верна: выборка N(-1,2), F0=N(-1,2): n=10, Dₙ=0.9429, α=0.9444
1b) H0 неверна: выборка N(-1,2), F0=L(-1,1): n=10, Dₙ=0.9944, α=0.9055


2. Объём $n = 50$, выборка из нормального распределения $\mathcal{N}(-1,2)$,
- распределение основной гипотезы $H_0$ - нормальное распределение $\mathcal{N}(-1,2)$,
- распределение основной гипотезы $H_0$ - распределение Лапласа $L(-1,1)$ с плотностью:
$$
    p(x)
    = \frac{1}{2} e^{-|x+1|}.
$$
Использовать асимптотическое распределение Колмогорова.

In [134]:
F0_normal  = lambda x: normal_cdf(x, -1, 2)
F0_laplace = lambda x: laplace_cdf(x, -1, 1)

print("n=50, асимптотика K(t)")
run_case("2a) H0 верна: выборка N(-1,2), F0=N(-1,2)", 50, 'normal', F0_normal,
         kolmogorov_cdf_asymp)
run_case("2b) H0 неверна: выборка N(-1,2), F0=L(-1,1)", 50, 'normal', F0_laplace,
         kolmogorov_cdf_asymp)

n=50, асимптотика K(t)
2a) H0 верна: выборка N(-1,2), F0=N(-1,2): n=50, Dₙ=1.1739, α=0.1271
2b) H0 неверна: выборка N(-1,2), F0=L(-1,1): n=50, Dₙ=1.4800, α=0.0250
